# Práctica — Clase 10. Rainbow y el aprendizaje distribucional

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

En esta práctica programas tú misma o tú mismo dos piezas de Rainbow sobre el DQN de la clase siete:

1. La arquitectura dueling completa: en la clase la viste ya montada, hoy la construyes desde cero
2. Los retornos multi-paso: la mejora que el estudio de ablación señaló como una de las que más suma al rendimiento final

Usa el mismo entorno CartPole-v1 y el mismo bucle de entrenamiento de la clase.

**Antes de empezar, ejecuta las celdas de la sección "Preparación".** No hace falta que las edites.


## Preparación

Instala gymnasium y define las piezas que ya conoces de la clase: el buffer de repetición, la red del DQN estándar (que actúa como referencia) y el bucle de entrenamiento completo.


In [ ]:
!pip install gymnasium


In [ ]:
import random
from collections import deque, namedtuple

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Fijamos semillas para que el experimento sea reproducible
SEMILLA = 0
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)


In [ ]:
Transicion = namedtuple(
    "Transicion", ["estado", "accion", "recompensa", "siguiente_estado", "terminado"]
)


class BufferReplay:
    """Memoria de repeticion de experiencia, identica a la de la clase."""

    def __init__(self, capacidad=10000):
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, *args):
        self.memoria.append(Transicion(*args))

    def muestrear(self, tamano_lote):
        return random.sample(self.memoria, tamano_lote)

    def __len__(self):
        return len(self.memoria)


class RedQEstandar(nn.Module):
    """DQN "de manual": una sola cabeza que produce directamente
    un valor Q por cada accion posible. La usas como linea base
    en el ejercicio uno y como red interna del ejercicio dos."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, x):
        return self.red(x)


def entrenar(clase_red, nombre, n_episodios=100, gamma=0.99, tamano_lote=64,
             tasa_aprendizaje=1e-3, actualizar_objetivo_cada=10):
    """Bucle de entrenamiento DQN completo, identico al de la clase."""
    entorno = gym.make("CartPole-v1")
    n_entradas = entorno.observation_space.shape[0]
    n_acciones = entorno.action_space.n

    red_online = clase_red(n_entradas, n_acciones)
    red_objetivo = clase_red(n_entradas, n_acciones)
    red_objetivo.load_state_dict(red_online.state_dict())
    red_objetivo.eval()

    optimizador = optim.Adam(red_online.parameters(), lr=tasa_aprendizaje)
    buffer = BufferReplay(capacidad=10000)

    epsilon, epsilon_minimo, decaimiento = 1.0, 0.05, 0.97
    recompensas_por_episodio = []

    for episodio in range(n_episodios):
        estado, _ = entorno.reset(seed=SEMILLA + episodio)
        recompensa_total = 0.0
        terminado = False

        while not terminado:
            if random.random() < epsilon:
                accion = entorno.action_space.sample()
            else:
                with torch.no_grad():
                    q_valores = red_online(torch.tensor(estado, dtype=torch.float32))
                    accion = int(torch.argmax(q_valores).item())

            siguiente_estado, recompensa, fin, truncado, _ = entorno.step(accion)
            terminado = fin or truncado
            buffer.guardar(estado, accion, recompensa, siguiente_estado, fin)
            estado = siguiente_estado
            recompensa_total += recompensa

            if len(buffer) >= tamano_lote:
                lote = buffer.muestrear(tamano_lote)
                estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
                acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64)
                recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
                siguientes = torch.tensor(np.array([t.siguiente_estado for t in lote]), dtype=torch.float32)
                terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

                q_actuales = red_online(estados).gather(1, acciones.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    q_siguientes = red_objetivo(siguientes).max(dim=1)[0]
                    objetivo = recompensas + gamma * q_siguientes * (1 - terminados)

                perdida = nn.functional.mse_loss(q_actuales, objetivo)
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()

        recompensas_por_episodio.append(recompensa_total)
        epsilon = max(epsilon_minimo, epsilon * decaimiento)

        if episodio % actualizar_objetivo_cada == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    entorno.close()
    print(f"{nombre}: recompensa media de los ultimos 20 episodios = "
          f"{np.mean(recompensas_por_episodio[-20:]):.1f}")
    return recompensas_por_episodio


### Entrenamos la línea base

**Ejecuta esta celda antes de empezar los ejercicios.** Vas a usar `recompensas_dqn_estandar` como punto de comparación en el ejercicio uno. Tarda aproximadamente medio minuto.


In [ ]:
recompensas_dqn_estandar = entrenar(RedQEstandar, "DQN estándar (línea base)")


## Ejercicio 1 — Completa la arquitectura dueling desde cero

En la clase viste la arquitectura dueling ya construida. Ahora te toca a ti: completa `RedQDuelingIncompleta`, donde pone `# TODO`.

Recuerda la idea: un tronco compartido alimenta dos cabezas.

- La **cabeza de valor** estima V(s): cuánto vale el estado en general, un único número
- La **cabeza de ventaja** estima A(s,a): cuánto mejor es cada acción concreta frente a las demás, un número por cada acción posible

Al combinarlas, la fórmula es:

**Q(s,a) = V(s) + (A(s,a) − media_a[A(s,a)])**

**Pistas**:

1. Cada cabeza es un `nn.Sequential` pequeño: una capa lineal oculta seguida de `nn.ReLU()`, y después una capa lineal final
2. La cabeza de valor termina en una sola neurona (`nn.Linear(n_ocultas, 1)`); la cabeza de ventaja termina en `n_acciones` neuronas
3. Para restar la media de la ventaja a lo largo de las acciones, usa `ventaja.mean(dim=-1, keepdim=True)`

**Tu tarea es completar los tres `# TODO`.**


In [ ]:
class RedQDuelingIncompleta(nn.Module):
    """Arquitectura dueling: un tronco compartido alimenta dos cabezas.
    Completa las dos cabezas y la formula de combinacion siguiendo
    las pistas del enunciado."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.tronco = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
        )

        # TODO 1: cabeza de valor V(s) -- debe terminar en una sola neurona
        self.cabeza_valor = None

        # TODO 2: cabeza de ventaja A(s,a) -- debe terminar en n_acciones neuronas
        self.cabeza_ventaja = None

    def forward(self, x):
        caracteristicas = self.tronco(x)
        valor = self.cabeza_valor(caracteristicas)
        ventaja = self.cabeza_ventaja(caracteristicas)

        # TODO 3: combina valor y ventaja: Q(s,a) = V(s) + (A(s,a) - media_a[A(s,a)])
        return None


**Ejecuta la celda siguiente para entrenar tu Dueling DQN y compararlo con la línea base.** Tarda aproximadamente medio minuto.


In [ ]:
recompensas_dueling = entrenar(RedQDuelingIncompleta, "Dueling DQN (tu implementacion)")

plt.figure(figsize=(10, 5))
plt.plot(recompensas_dqn_estandar, label="DQN estandar (linea base)", alpha=0.6)
plt.plot(recompensas_dueling, label="Dueling DQN (tu implementacion)", alpha=0.6)
plt.xlabel("Episodio")
plt.ylabel("Recompensa")
plt.title("CartPole-v1: DQN estandar vs. tu Dueling DQN")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Ver solución</summary>

```python
class RedQDuelingIncompleta(nn.Module):
    """Arquitectura dueling: un tronco compartido alimenta dos cabezas.
    Una estima V(s) -- cuánto vale el estado, un único número --
    y otra estima A(s,a) -- cuánto mejor es cada acción frente a las demás."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.tronco = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
        )

        # Cabeza de valor V(s): termina en una sola neurona
        self.cabeza_valor = nn.Sequential(
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, 1),
        )

        # Cabeza de ventaja A(s,a): termina en n_acciones neuronas
        self.cabeza_ventaja = nn.Sequential(
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, x):
        caracteristicas = self.tronco(x)
        valor = self.cabeza_valor(caracteristicas)
        ventaja = self.cabeza_ventaja(caracteristicas)
        # Q(s,a) = V(s) + (A(s,a) - media_a[A(s,a)])
        return valor + (ventaja - ventaja.mean(dim=-1, keepdim=True))
```

En CartPole es normal que el Dueling DQN no siempre supere a la línea base: es un entorno con pocos estados donde casi todas las acciones ya influyen en el resultado, así que la arquitectura dueling no tiene tanto margen para lucirse como en entornos más grandes.
</details>


## Ejercicio 2 — Añade retornos multi-paso al bucle de entrenamiento

El DQN estándar mira un solo paso hacia delante antes de actualizar sus estimaciones. Ahora vas a acumular la recompensa de varios pasos seguidos (n = 3) antes de bootstrapear con la red objetivo, y vas a comparar la curva resultante con la del DQN de un solo paso.

Abajo tienes `entrenar_nstep_incompleto`, una copia del bucle de entrenamiento que ya mantiene una ventana `ventana_nstep` con los últimos n pasos —estado, acción, recompensa— y que ya sabe bootstrapear con `gamma ** n_pasos` una vez tiene el retorno acumulado. **Lo único que falta es calcular ese retorno.**

**Pistas**:

1. `ventana_nstep` es un `deque` con hasta `n_pasos` transiciones `(estado, accion, recompensa)`, ordenadas de la más antigua a la más nueva
2. El retorno multi-paso descuenta cada recompensa según su posición dentro de la ventana: la primera sin descuento (gamma elevado a cero), la segunda multiplicada por gamma, la tercera por gamma al cuadrado, y así sucesivamente
3. Recorre `ventana_nstep` con `enumerate` para obtener, a la vez, la posición `i` y la recompensa `r` de cada transición

**Tu tarea es completar el `# TODO`.**


In [ ]:
def entrenar_nstep_incompleto(nombre, n_pasos=1, n_episodios=100, gamma=0.99, tamano_lote=64,
                               tasa_aprendizaje=1e-3, actualizar_objetivo_cada=10):
    """Bucle de entrenamiento DQN con retornos multi-paso. Con n_pasos=1
    equivale al DQN estandar de la seccion de preparacion."""
    entorno = gym.make("CartPole-v1")
    n_entradas = entorno.observation_space.shape[0]
    n_acciones = entorno.action_space.n

    red_online = RedQEstandar(n_entradas, n_acciones)
    red_objetivo = RedQEstandar(n_entradas, n_acciones)
    red_objetivo.load_state_dict(red_online.state_dict())
    red_objetivo.eval()

    optimizador = optim.Adam(red_online.parameters(), lr=tasa_aprendizaje)
    buffer = BufferReplay(capacidad=10000)

    epsilon, epsilon_minimo, decaimiento = 1.0, 0.05, 0.97
    recompensas_por_episodio = []

    for episodio in range(n_episodios):
        estado, _ = entorno.reset(seed=SEMILLA + episodio)
        recompensa_total = 0.0
        terminado = False
        ventana_nstep = deque(maxlen=n_pasos)  # guarda (estado, accion, recompensa)

        while not terminado:
            if random.random() < epsilon:
                accion = entorno.action_space.sample()
            else:
                with torch.no_grad():
                    q_valores = red_online(torch.tensor(estado, dtype=torch.float32))
                    accion = int(torch.argmax(q_valores).item())

            siguiente_estado, recompensa, fin, truncado, _ = entorno.step(accion)
            terminado = fin or truncado
            ventana_nstep.append((estado, accion, recompensa))
            estado = siguiente_estado
            recompensa_total += recompensa

            # En marcha normal emitimos una transicion n-step por paso; al terminar
            # el episodio vaciamos toda la ventana que quede pendiente.
            while len(ventana_nstep) == n_pasos or (terminado and len(ventana_nstep) > 0):
                estado_inicial, accion_inicial, _ = ventana_nstep[0]

                # TODO: calcula el retorno acumulado de las recompensas que quedan en
                # `ventana_nstep`, descontando cada una segun su posicion (gamma**i),
                # antes de que el bootstrap de mas abajo aplique el descuento final
                # gamma**n_pasos sobre el valor del estado siguiente
                retorno_nstep = None

                buffer.guardar(
                    estado_inicial, accion_inicial, retorno_nstep,
                    siguiente_estado, fin,
                )
                ventana_nstep.popleft()

            if len(buffer) >= tamano_lote:
                lote = buffer.muestrear(tamano_lote)
                estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
                acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64)
                recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
                siguientes = torch.tensor(np.array([t.siguiente_estado for t in lote]), dtype=torch.float32)
                terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

                q_actuales = red_online(estados).gather(1, acciones.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    q_siguientes = red_objetivo(siguientes).max(dim=1)[0]
                    objetivo = recompensas + (gamma ** n_pasos) * q_siguientes * (1 - terminados)

                perdida = nn.functional.mse_loss(q_actuales, objetivo)
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()

        recompensas_por_episodio.append(recompensa_total)
        epsilon = max(epsilon_minimo, epsilon * decaimiento)

        if episodio % actualizar_objetivo_cada == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    entorno.close()
    print(f"{nombre}: recompensa media de los ultimos 20 episodios = "
          f"{np.mean(recompensas_por_episodio[-20:]):.1f}")
    return recompensas_por_episodio


**Ejecuta la celda siguiente para comparar un paso frente a tres pasos.** Tarda aproximadamente un minuto en total.


In [ ]:
recompensas_1paso = entrenar_nstep_incompleto("DQN 1 paso", n_pasos=1)
recompensas_3pasos = entrenar_nstep_incompleto("DQN 3 pasos", n_pasos=3)

plt.figure(figsize=(10, 5))
plt.plot(recompensas_1paso, label="DQN 1 paso", alpha=0.6)
plt.plot(recompensas_3pasos, label="DQN 3 pasos", alpha=0.6)
plt.xlabel("Episodio")
plt.ylabel("Recompensa")
plt.title("CartPole-v1: retornos de un paso vs. retornos multi-paso (n=3)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Ver solución</summary>

```python
            while len(ventana_nstep) == n_pasos or (terminado and len(ventana_nstep) > 0):
                estado_inicial, accion_inicial, _ = ventana_nstep[0]

                # Sumamos cada recompensa de la ventana descontada segun su posicion:
                # la mas antigua (i=0) sin descuento, la siguiente por gamma, etc.
                retorno_nstep = 0.0
                for i, (_, _, r) in enumerate(ventana_nstep):
                    retorno_nstep += (gamma ** i) * r

                buffer.guardar(
                    estado_inicial, accion_inicial, retorno_nstep,
                    siguiente_estado, fin,
                )
                ventana_nstep.popleft()
```

Con `n_pasos=3` el agente suele aprender más rápido que con un solo paso: cada actualización ya incorpora tres recompensas reales en vez de una, así que la señal de aprendizaje llega con menos ruido y el bootstrap con la red objetivo pesa menos en el resultado.
</details>


## Para seguir practicando

**Cambia `n_pasos` a cinco en el ejercicio dos y vuelve a ejecutar la comparación.** ¿La curva sigue mejorando cuanto más lejos miras el agente, o en algún punto empieza a empeorar?

Comparte tu resultado con tu equipo y **compara** qué combinación de mejoras dio la curva de aprendizaje más estable.
